# 07 — BM25 / Dense / Hybrid baselines v2 (BM25 tokenizer fix)

`src/utils.py`-e `tokenize()` (punctuation strip) already push kora ache. Ei notebook shob first-stage retriever abar chalay (LLM chhara) ar v1 (purano) result-er shathe tulona kore.
Per-query score save kore `results/baselines_v2_perquery.jsonl`-e → pore compound-density analysis-e lagbe.

**Kaggle:** Accelerator GPU, Internet On. Cell 1 theke serially chalao.

In [1]:
!pip install datasets rank_bm25 sentence-transformers -q

In [2]:
import os, sys, subprocess, shutil, glob

REPO_URL = "https://github.com/medhuuuu/swedish-retrieval-project.git"
REPO_DIR = "/kaggle/working/swedish-retrieval-project"
if os.path.exists(REPO_DIR):
    print(subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True).stdout)
else:
    print(subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True))
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

BASE = "/kaggle/working/swedish_retrieval"
CACHE_DIR = os.path.join(BASE, "cache")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(os.path.join(REPO_DIR, "results"), exist_ok=True)
for f in glob.glob("/kaggle/input/**/*.npy", recursive=True):   # ager session-er embedding thakle restore
    dst = os.path.join(CACHE_DIR, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst); print("restored", os.path.basename(f))

# check: notun utils-e tokenize ache ki na
from src.utils import tokenize
print(tokenize("Får jag lägre föräldrapenning innan förlossningen?"))

Already up to date.

['får', 'jag', 'lägre', 'föräldrapenning', 'innan', 'förlossningen']


In [3]:
import numpy as np, json, pandas as pd, torch
from src.utils import (load_swefaq, load_swedn, build_qrels_dict, build_bm25, tokenize,
                       build_chunked_corpus, search_dense, dense_ranked_ids_chunked,
                       search_hybrid, search_hybrid_chunked, recall_at_k, ndcg_at_k)

swefaq_corpus, swefaq_queries, swefaq_qrels = load_swefaq()
swedn_corpus, swedn_queries, swedn_qrels = load_swedn()
swefaq_qrels_dict = build_qrels_dict(swefaq_qrels)
swedn_qrels_dict = build_qrels_dict(swedn_qrels)

swefaq_bm25, swefaq_corpus_ids, swefaq_corpus_texts = build_bm25(swefaq_corpus)
swedn_bm25, swedn_corpus_ids, swedn_corpus_texts = build_bm25(swedn_corpus)
print("SweFAQ", len(swefaq_corpus_ids), "docs |", "Swedn", len(swedn_corpus_ids), "docs")

README.md:   0%|          | 0.00/6.41k [00:00<?, ?B/s]

corpus/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 88.9kB            

corpus/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/511 [00:00<?, ? examples/s]

queries/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 23.5kB            

queries/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (38 outputs left out)

Map:   0%|          | 0/2046 [00:00<?, ? examples/s]

Map:   0%|          | 0/1024 [00:00<?, ? examples/s]

Map:   0%|          | 0/2048 [00:00<?, ? examples/s]

SweFAQ 511 docs | Swedn 6141 docs


In [4]:
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda:0" if torch.cuda.is_available() else "cpu")
emb_model.max_seq_length = 512

def cached_encode(name, texts, prefix="passage: "):
    path = os.path.join(CACHE_DIR, f"{name}.npy")
    if os.path.exists(path):
        arr = np.load(path)
        if len(arr) == len(texts):
            print(f"[cache hit] {name} {arr.shape}"); return arr
    arr = emb_model.encode([prefix + t for t in texts], batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(path, arr); return arr

swefaq_embeddings = cached_encode("swefaq_passages", swefaq_corpus_texts)
swedn_embeddings = cached_encode("swedn_passages", swedn_corpus_texts)
swedn_chunk_ids, swedn_chunk_texts, swedn_parent_ids = build_chunked_corpus(swedn_corpus)
swedn_chunk_embeddings = cached_encode("swedn_chunks_400_50", swedn_chunk_texts)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

... (7 outputs left out)

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/96 [00:00<?, ?it/s]

Batches:   0%|          | 0/159 [00:00<?, ?it/s]

In [5]:
def bm25_fn(bm25, ids):
    def f(q, k):
        s = bm25.get_scores(tokenize(q))     # <-- tokenize (purano .lower().split() na)
        return [ids[i] for i in s.argsort()[::-1][:k]]
    return f

experiments = {
    "swefaq_bm25":   ("swefaq", bm25_fn(swefaq_bm25, swefaq_corpus_ids)),
    "swefaq_dense":  ("swefaq", lambda q, k: search_dense(q, emb_model, swefaq_embeddings, swefaq_corpus_ids, top_k=k)),
    "swefaq_hybrid": ("swefaq", lambda q, k: search_hybrid(q, swefaq_bm25, swefaq_corpus_ids, swefaq_embeddings, emb_model, top_k=k)),
    "swedn_bm25":           ("swedn", bm25_fn(swedn_bm25, swedn_corpus_ids)),
    "swedn_dense_trunc":    ("swedn", lambda q, k: search_dense(q, emb_model, swedn_embeddings, swedn_corpus_ids, top_k=k)),
    "swedn_dense_chunked":  ("swedn", lambda q, k: dense_ranked_ids_chunked(q, emb_model, swedn_chunk_embeddings, swedn_parent_ids, pool_size=k)),
    "swedn_hybrid_trunc":   ("swedn", lambda q, k: search_hybrid(q, swedn_bm25, swedn_corpus_ids, swedn_embeddings, emb_model, top_k=k)),
    "swedn_hybrid_chunked": ("swedn", lambda q, k: search_hybrid_chunked(q, swedn_bm25, swedn_corpus_ids, swedn_chunk_embeddings, swedn_parent_ids, emb_model, top_k=k)),
}
data = {"swefaq": (swefaq_queries, swefaq_qrels_dict), "swedn": (swedn_queries, swedn_qrels_dict)}

# sanity
for n, (ds, fn) in experiments.items():
    assert len(fn(data[ds][0][0]["text"], 3)) == 3, n
print("OK")

OK


## Evaluate (8 experiment) + per-query save

In [6]:
perq_path = os.path.join(REPO_DIR, "results", "baselines_v2_perquery.jsonl")
rows = []
with open(perq_path, "w") as fout:
    for name, (ds, fn) in experiments.items():
        qs, qr = data[ds]
        m = {"r1": [], "r3": [], "r10": [], "ndcg10": []}
        for q in qs:
            if q["_id"] not in qr:
                continue
            ranked = fn(q["text"], 10)
            rel = qr[q["_id"]]
            rec = {"r1": recall_at_k(ranked, rel, 1), "r3": recall_at_k(ranked, rel, 3),
                   "r10": recall_at_k(ranked, rel, 10), "ndcg10": ndcg_at_k(ranked, rel, 10)}
            for k, v in rec.items():
                m[k].append(v)
            fout.write(json.dumps({"exp": name, "qid": q["_id"], "query": q["text"], **rec}) + "\n")
        rows.append({"name": name, "n_queries": len(m["r1"]), **{k: float(np.mean(v)) for k, v in m.items()}})
v2 = pd.DataFrame(rows)
print("saved", perq_path)

saved /kaggle/working/swedish-retrieval-project/results/baselines_v2_perquery.jsonl


## v1 (purano, `.lower().split()`) vs v2 (notun, `tokenize`)

In [7]:
v1 = pd.DataFrame([
    ("swefaq_bm25", .2320, .3957, .5887, .3957), ("swefaq_dense", .6062, .7836, .9025, .7528),
    ("swefaq_hybrid", .4561, .6238, .7271, .5962), ("swedn_bm25", .2980, .5122, .6149, .5709),
    ("swedn_dense_trunc", .3807, .7264, .8229, .7704), ("swedn_dense_chunked", .3840, .7362, .8286, .7769),
    ("swedn_hybrid_trunc", .3558, .6354, .7718, .7033), ("swedn_hybrid_chunked", .3626, .6406, .7699, .7072),
], columns=["name", "r1_v1", "r3_v1", "r10_v1", "ndcg10_v1"])
cmp = v2.rename(columns={"r1": "r1_v2", "r3": "r3_v2", "r10": "r10_v2", "ndcg10": "ndcg10_v2"}).merge(v1, on="name")
cmp["ndcg10_delta"] = cmp["ndcg10_v2"] - cmp["ndcg10_v1"]
cols = ["name", "n_queries", "r1_v1", "r1_v2", "r3_v1", "r3_v2", "r10_v1", "r10_v2", "ndcg10_v1", "ndcg10_v2", "ndcg10_delta"]
cmp[cols].to_csv(os.path.join(REPO_DIR, "results", "baselines_v1_vs_v2.csv"), index=False)
cmp[cols].round(4)

,name,n_queries,r1_v1,r1_v2,r3_v1,r3_v2,r10_v1,r10_v2,ndcg10_v1,ndcg10_v2,ndcg10_delta
0,swefaq_bm25,513,0.2320,0.3450,0.3957,0.5400,0.5887,0.7115,0.3957,0.5216,0.1259
1,swefaq_dense,513,0.6062,0.6062,0.7836,0.7836,0.9025,0.9025,0.7528,0.7528,-0.0000
2,swefaq_hybrid,513,0.4561,0.5205,0.6238,0.7018,0.7271,0.8226,0.5962,0.6725,0.0763
3,swedn_bm25,3072,0.2980,0.3576,0.5122,0.6268,0.6149,0.7174,0.5709,0.6803,0.1094
4,swedn_dense_trunc,3072,0.3807,0.3807,0.7264,0.7264,0.8229,0.8229,0.7704,0.7704,0.0000
5,swedn_dense_chunked,3072,0.3840,0.3840,0.7362,0.7362,0.8286,0.8286,0.7769,0.7769,-0.0000
6,swedn_hybrid_trunc,3072,0.3558,0.3807,0.6354,0.6966,0.7718,0.8035,0.7033,0.7504,0.0471
7,swedn_hybrid_chunked,3072,0.3626,0.3892,0.6406,0.7041,0.7699,0.8044,0.7072,0.7573,0.0501
